In [1]:
import os
import sys
sys.path.append(os.path.join('c:\\', *os.getcwd().split('\\')[1:-1]))
from sgcc import *

In [2]:
param_bounds_1 = {
    "fts": [0, 0],
    "t": [70, 70],
    "ampc": [0.01, 0.1],
    'ampm': [0,3],
    "ampg": [0.1, 4],
    "ampw": [0.01,0.08],
    "d": [10, 40],
    "inh_d": [0, 40],
    "inh_w": [0, 3],
}

param_bounds_2 = {
    "fts": [0, 0],
    "t": [80, 80],
    "ampc": [0.01, 0.1],
    'ampm': [0,3],
    "ampg": [0.1, 4],
    "ampw": [0.01,0.08],
    "d": [10, 40],
    "inh_d": [0, 40],
    "inh_w": [0, 3],
}

In [3]:
bounds = [
    param_bounds_1,
    param_bounds_2
]

In [4]:
model = SGCCircuit(bounds=param_bounds_1)

In [5]:
model.initialize_random_parameters(n_v1=2, n_lgn=3, n_sample=100)

In [6]:
dlgn_bounds = [
    tf.convert_to_tensor([x for x in inner_bounds.values()])[:-2]
    for inner_bounds in bounds
]
dlgn_lower = [
    tf.reshape(inner_bounds[:,0], [1,1,-1,1,1])
    for inner_bounds in dlgn_bounds
]
dlgn_upper = [
    tf.reshape(inner_bounds[:,1], [1,1,-1,1,1])
    for inner_bounds in dlgn_bounds
]

v1_bounds = [
    tf.convert_to_tensor([x for x in inner_bounds.values()])[-2:]
    for inner_bounds in bounds
]
v1_lower = [
    tf.reshape(inner_bounds[:,0], [1,1,-1,1,1])
    for inner_bounds in v1_bounds
]
v1_upper = [
    tf.reshape(inner_bounds[:,1], [1,1,-1,1,1])
    for inner_bounds in v1_bounds
]

dlgn_unstacked = []
v1_unstacked = []
for i in range(model.n_v1):
    
    dlgn_unstacked.append(
        model.variable_transformer(
        model.dlgn_scaled[:,i,:,:,:,:], 
        dlgn_lower[i], 
        dlgn_upper[i]
        )
    )

    v1_unstacked.append(
        model.variable_transformer(
        model.v1_scaled[:,i,:,:,:,:], 
        v1_lower[i], 
        v1_upper[i]
        )
    )

model.params = {
    'dLGN_params': tf.stack(dlgn_unstacked, axis=1),
    'V1_params': tf.stack(v1_unstacked, axis=1)
}

In [7]:
def update_transform_separate(self):

    ## Extract dlgn bounds
    dlgn_bounds = [
        tf.convert_to_tensor([x for x in inner_bounds.values()])[:-2]
        for inner_bounds in self.bounds
    ]
    dlgn_lower = [
        tf.reshape(inner_bounds[:,0], [1,1,-1,1,1])
        for inner_bounds in dlgn_bounds
    ]
    dlgn_upper = [
        tf.reshape(inner_bounds[:,1], [1,1,-1,1,1])
        for inner_bounds in dlgn_bounds
    ]

    # Extract v1 bounds
    v1_bounds = [
        tf.convert_to_tensor([x for x in inner_bounds.values()])[-2:]
        for inner_bounds in self.bounds
    ]
    v1_lower = [
        tf.reshape(inner_bounds[:,0], [1,1,-1,1,1])
        for inner_bounds in v1_bounds
    ]
    v1_upper = [
        tf.reshape(inner_bounds[:,1], [1,1,-1,1,1])
        for inner_bounds in v1_bounds
    ]

    dlgn_unstacked = []
    v1_unstacked = []
    for i in range(self.n_v1):
        
        dlgn_unstacked.append(
            self.variable_transformer(
            self.dlgn_scaled[:,i,:,:,:,:], 
            dlgn_lower[i], 
            dlgn_upper[i]
            )
        )

        v1_unstacked.append(
            self.variable_transformer(
            self.v1_scaled[:,i,:,:,:,:], 
            v1_lower[i], 
            v1_upper[i]
            )
        )

    self.params = {
        'dLGN_params': tf.stack(dlgn_unstacked, axis=1),
        'V1_params': tf.stack(v1_unstacked, axis=1)
    }

In [8]:
model = SGCCircuit(bounds=bounds)

In [9]:
model.initialize_random_parameters(n_v1=2, n_lgn=3, n_sample=100)

In [10]:
len(model.bounds)

2